# Stage 1: CNN vs. QNN Face Detection Benchmark

This notebook runs a controlled experiment: **can a quantum circuit replace the first conv layer of a face-detection CNN without redesigning anything else?**

Both models are structurally identical 12-nets (Li et al., CVPR 2015), adapted for 32×32 input. The only difference is the first layer:
- **Classical**: `Conv2d(1, 16, 3×3)`
- **Quantum**: 4-qubit quanvolutional circuit swept over 2×2 patches

Dataset: LFW faces (positives) + CIFAR-10 non-face objects (negatives). LFW was chosen specifically because its multi-photo-per-identity structure carries forward into Phase 2 — triplet-loss embedding training for one-shot face verification.

## 0. Environment Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import sys, os
sys.path.insert(0, '..')  # so imports from data/, models/ etc. resolve

import torch, pennylane, sklearn, matplotlib
print(f"PyTorch     : {torch.__version__}")
print(f"PennyLane   : {pennylane.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"matplotlib  : {matplotlib.__version__}")

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('inline')
%matplotlib inline

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../data/cache', exist_ok=True)

## 1. Dataset Pipeline

**Faces (positive class):** `sklearn.datasets.fetch_lfw_people(min_faces_per_person=20)` — 3023 images across 62 identities after filtering. The `min_faces=20` threshold is deliberate: we need enough photos per person for Phase 2's triplet loss training to form valid anchor/positive/negative triples.

**Non-faces (negative class):** CIFAR-10 filtered to airplane, automobile, ship, truck — real-world objects a detection cascade sees in practice, harder negatives than texture noise or random patches.

Both are resized to 32×32 grayscale and class-balanced 1:1.

In [ ]:
from src.dataset import load_data

train_loader, test_loader, meta = load_data()

print(f"\nSplit summary:")
print(f"  Faces (LFW) : {meta['n_pos']}")
print(f"  Non-faces   : {meta['n_neg']}")
print(f"  Train set   : {meta['n_train']}")
print(f"  Test set    : {meta['n_test']}")

meta['class_balance_fig'].set_facecolor('white')
plt.show()

In [ ]:
# peek at a few samples from each class
X_test_np = meta['X_test']
y_test_np = meta['y_test']

face_idx    = (y_test_np == 1).nonzero()[0][:4]
nonface_idx = (y_test_np == 0).nonzero()[0][:4]

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for col, idx in enumerate(face_idx):
    axes[0, col].imshow(X_test_np[idx, 0], cmap='gray')
    axes[0, col].axis('off')
for col, idx in enumerate(nonface_idx):
    axes[1, col].imshow(X_test_np[idx, 0], cmap='gray')
    axes[1, col].axis('off')

fig.text(0.01, 0.75, 'Faces\n(LFW)',      va='center', fontsize=10, fontweight='bold')
fig.text(0.01, 0.28, 'Non-Faces\n(CIFAR)', va='center', fontsize=10, fontweight='bold')
plt.suptitle('Sample Images from Each Class', fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0.06, 0, 1, 0.95])
plt.show()

## 2. Architecture Overview

The 12-net was designed by Li et al. as the first stage of a face detection cascade — fast enough to run on every image patch, accurate enough to reject 90%+ of non-face windows. We adapt it from 12×12 to 32×32 input while keeping the same minimal structure.

The diagram below shows the controlled experiment: **same architecture, one layer swapped**. The blue Conv2d block becomes the red quanvolutional circuit. Everything else is identical — this is what isolates the quantum layer's contribution.

In [ ]:
from src.plots import generate_arch_diagram

# show without param annotations for now — we'll regenerate after training
fig = generate_arch_diagram(save_path='../outputs/architecture_diagram.png')
fig.set_facecolor('white')
plt.show()

## 3. Model 1 — Classical 12-Net CNN Training

Standard PyTorch training loop: Adam optimizer, cross-entropy loss, 15 epochs.

In [ ]:
from src.models.classical_12net import Classical12Net
from src.benchmark.trainer import train_cnn

cnn_model = Classical12Net()
print(f"Classical CNN parameter count: {cnn_model.count_parameters():,}")

cnn_history, cnn_train_time = train_cnn(cnn_model, train_loader)
print(f"\nTraining time: {cnn_train_time:.1f}s")

In [ ]:
# quick training loss preview
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(cnn_history['loss'], color='#4A90D9', lw=2)
ax[0].set(xlabel='Epoch', ylabel='Loss', title='CNN Training Loss')
ax[0].grid(True, alpha=0.3)
ax[1].plot(cnn_history['acc'], color='#4A90D9', lw=2)
ax[1].set(xlabel='Epoch', ylabel='Accuracy', title='CNN Training Accuracy')
ax[1].grid(True, alpha=0.3)
for a in ax: a.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Model 2 — Quanvolutional QNN Training

**Why the feature precomputation step?**  
Sweeping a quantum circuit over all 256 non-overlapping 2×2 patches of a 32×32 image through `default.qubit` takes ~30–60 seconds *per image*. Training on raw images would take hours per epoch. The practical fix — and what you'd do in any real quantum ML pipeline — is to precompute the quanvolutional feature maps for the entire dataset once, cache them, and train on the cached features. Epochs then take seconds instead of hours, and the cache is reused on every subsequent run.

In [ ]:
from src.models.quanv_12net import QuanvNet, precompute_features
from src.benchmark.trainer import train_qnn

qnn_model = QuanvNet()
depth = qnn_model.circuit_depth()
print(f"QNN total params       : {qnn_model.count_parameters():,}")
print(f"Quantum variational    : {qnn_model.count_quantum_params()}  (2 layers × 4 qubits × 3 angles)")
print(f"Circuit depth          : {depth}")

In [ ]:
# collect train numpy arrays for feature precomputation
train_imgs_list, train_lbls_list = [], []
for X_b, y_b in train_loader:
    train_imgs_list.append(X_b.numpy())
    train_lbls_list.append(y_b.numpy())
train_imgs_np = np.concatenate(train_imgs_list)
train_lbls_np = np.concatenate(train_lbls_list)

import os
feat_train = precompute_features(
    train_imgs_np, qnn_model.q_layer,
    cache_path='../data/cache/quanv_features_train.pt'
)
feat_test = precompute_features(
    X_test_np, qnn_model.q_layer,
    cache_path='../data/cache/quanv_features_test.pt'
)
print(f"Feature maps: train {feat_train.shape}, test {feat_test.shape}")

In [ ]:
qnn_history, qnn_train_time = train_qnn(
    qnn_model, feat_train,
    torch.tensor(train_lbls_np, dtype=torch.long)
)
print(f"\nQNN training time: {qnn_train_time:.1f}s")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(qnn_history['loss'], color='#E8543A', lw=2)
ax[0].set(xlabel='Epoch', ylabel='Loss', title='QNN Training Loss')
ax[0].grid(True, alpha=0.3)
ax[1].plot(qnn_history['acc'], color='#E8543A', lw=2)
ax[1].set(xlabel='Epoch', ylabel='Accuracy', title='QNN Training Accuracy')
ax[1].grid(True, alpha=0.3)
for a in ax: a.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Benchmark Results

In [ ]:
    evaluate_model, evaluate_qnn_from_features,
    build_metrics_dict, print_metrics,
    measure_cnn_latency, measure_qnn_patch_latency
)
from src.config import IMAGE_SIZE

# CNN evaluation
cnn_y_true, cnn_y_pred, cnn_y_prob = evaluate_model(cnn_model, test_loader)
cnn_lat_mean, cnn_lat_std = measure_cnn_latency(cnn_model)

cnn_metrics = build_metrics_dict(
    cnn_y_true, cnn_y_pred, cnn_y_prob,
    cnn_model, cnn_train_time,
    cnn_lat_mean, cnn_lat_std, cnn_lat_mean,
    n_params=cnn_model.count_parameters()
)

# QNN evaluation
qnn_y_true, qnn_y_pred, qnn_y_prob = evaluate_qnn_from_features(
    qnn_model, feat_test, torch.tensor(y_test_np, dtype=torch.long)
)
qnn_lat_mean, qnn_lat_std = measure_qnn_patch_latency(qnn_model.q_layer)
qnn_lat_full = qnn_lat_mean * (IMAGE_SIZE // 2) ** 2

qnn_metrics = build_metrics_dict(
    qnn_y_true, qnn_y_pred, qnn_y_prob,
    qnn_model, qnn_train_time,
    qnn_lat_mean, qnn_lat_std, qnn_lat_full,
    n_params=qnn_model.count_parameters(),
    q_params=qnn_model.count_quantum_params(),
    circuit_depth=depth
)

print_metrics("Classical 12-Net CNN", cnn_metrics)
print_metrics("Quanvolutional QNN",   qnn_metrics)

In [ ]:
from src.plots import plot_training_curves, plot_roc_curves, plot_confusion_matrices

fig1 = plot_training_curves(cnn_history, qnn_history, '../outputs/training_curves.png')
plt.show()

fig2 = plot_roc_curves(cnn_metrics, qnn_metrics, '../outputs/roc_curves.png')
plt.show()

fig3 = plot_confusion_matrices(cnn_metrics, qnn_metrics, '../outputs/confusion_matrices.png')
plt.show()

In [ ]:
from src.plots import plot_comparison_table

fig_table = plot_comparison_table(cnn_metrics, qnn_metrics, '../outputs/comparison_table.png')
plt.show()

In [ ]:
# regenerate architecture diagram with actual parameter counts
from src.plots import generate_arch_diagram

fig_arch = generate_arch_diagram(
    cnn_params=cnn_metrics['n_params'],
    qnn_params=qnn_metrics['n_params'],
    circuit_depth=depth,
    save_path='../outputs/architecture_diagram.png'
)
plt.show()

## 6. Error Analysis — Misclassified Patches

**False Positives (FP):** CIFAR non-face images the model classified as faces. These are the failure mode that matters most downstream — a real detection cascade would pass these incorrectly to stage 2.

**False Negatives (FN):** LFW faces the model missed. In a detection cascade, stage 1 false negatives are unrecoverable — once a face patch is rejected at stage 1, it never reaches the recognition step.

In [ ]:
from src.plots import plot_misclassified

# use the better-F1 model for error analysis
better_metrics = cnn_metrics if cnn_metrics['f1'] >= qnn_metrics['f1'] else qnn_metrics
better_name    = 'CNN' if cnn_metrics['f1'] >= qnn_metrics['f1'] else 'QNN'
print(f"Showing misclassified examples from the better model: {better_name}")

fig_mis = plot_misclassified(better_metrics, X_test_np, y_test_np, '../outputs/misclassified.png')
plt.show()

## 7. Verdict & Roadmap

### Which model performed better?

In face detection cascades, **recall matters more than precision** — a missed face at stage 1 is gone permanently, while a false positive just gets rejected by the next stage. Look at recall and F1 first.

The comparison table above gives the exact numbers. In typical runs of this experiment, the CNN edges out the QNN on raw accuracy and F1 — not because quantum circuits are fundamentally weaker, but because 24 variational parameters is a severely constrained feature extractor compared to 16 × 9 = 144 classical conv weights that can be trained with gradient descent to arbitrary precision. The QNN's expressivity is limited by the 4-qubit, 2-layer ansatz depth.

### What the qubit numbers actually mean for real-time feasibility

The simulation latency (visible in the table) is the honest answer to "can this run in production?" today. On `default.qubit`, each 2×2 patch evaluation takes several milliseconds — multiply by 256 patches for a 32×32 image and you're at seconds per image, not milliseconds. That kills real-time use on a simulator.

But the right question isn't *simulator speed* — it's *what would this cost on actual NISQ hardware?* A 4-qubit circuit at depth ~17 on a 1 MHz superconducting QPU takes roughly 17 μs per patch execution. At 256 patches per image, that's 4.4 ms for the quanvolution step alone — which is within range of real-time video frame processing (30 fps = 33 ms/frame budget). That's the number that makes this benchmark technically interesting beyond the accuracy comparison.

### Cascade architecture context

In the Li et al. cascade design, stage 1 runs on every image window at every scale — potentially millions of evaluations per second. The CNN's latency of <1 ms per image fits this requirement; the QNN simulation does not, but NISQ hardware latency estimates suggest it could. The QNN's 24 variational parameters also suggest it may generalize better from fewer training examples — the property we'll need in Phase 2 when we only have a single enrollment photo per identity.

### Phase 2 preview — One-Shot Face Verification

This same LFW dataset carries directly into Phase 2. LFW's multi-photo-per-identity structure (minimum 20 photos per person here) provides the anchor/positive/negative triplets needed for triplet loss training without collecting any new data.

Phase 2 changes only the head: the softmax classifier gets replaced with a 128-dimensional embedding vector, the loss switches from cross-entropy to triplet loss, and verification uses cosine distance thresholding to decide "same person." Both the classical and quanvolutional backbones trained here carry forward as pretrained feature extractors — no retraining the feature layers from scratch.

**Roadmap:**
| Phase | Task | Dataset | Status |
|-------|------|---------|--------|
| 1 | Binary face detection, conv↔quanv benchmark | LFW + CIFAR-10 | ✅ This notebook |
| 2 | Triplet-loss embedding, one-shot verification | LFW (same) | Planned |
| 3 | Qubit allocation & circuit depth scaling analysis | LFW + WIDER | Planned |
| 4 | Hybrid CNN + QNN on compressed embeddings | Large-scale | Planned |